In [8]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import huggingface_hub as hub
import datasets as hf


In [9]:
pd.read_csv("valid_messages.csv")

,Message ID,Timestamp,Sender / Source,Category,Message Content
0,1,NaN,SMS / Service Alert,General,"Dear Sir/Mam, We are happy to announce our pre..."
1,2,NaN,SMS / Service Alert,Financial / Transactional,Client Funds on 30/03/2024.MCX 0.Net Exchanges...
2,3,NaN,SMS / Service Alert,OTP / Security,"Dear Applicant, your one time password(OTP) is..."
3,4,NaN,SMS / Service Alert,OTP / Security,Your OTP for login into PPP is 291718. It is v...
4,5,NaN,SMS / Service Alert,OTP / Security,OTP for transaction in State Bank of India (SB...
...,...,...,...,...,...
60,61,NaN,SMS / Service Alert,OTP / Security,Your flipkart.com order is out for delivery. P...
61,62,NaN,SMS / Service Alert,Order / Delivery,"To ensure a seamless delivery process, confirm..."
62,63,NaN,SMS / Service Alert,Order / Delivery,Contact Delivery Partner: 01206491777 \nPIN: 161
63,64,"28/09/26, 7:48:28 PM",RAJDEEP KUMAR,Order / Delivery,Thank you for your response. The delivery will...


In [10]:
%pip install datasets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
import datasets as hf

df = hf.load_dataset("sidzzz07/scamshield-dataset")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'language', 'source_dataset', 'is_scam', 'head1_social_engineering', 'head2_scam_intent', 'split_group', 'metadata'],
        num_rows: 69286
    })
    validation: Dataset({
        features: ['id', 'text', 'language', 'source_dataset', 'is_scam', 'head1_social_engineering', 'head2_scam_intent', 'split_group', 'metadata'],
        num_rows: 8192
    })
    test: Dataset({
        features: ['id', 'text', 'language', 'source_dataset', 'is_scam', 'head1_social_engineering', 'head2_scam_intent', 'split_group', 'metadata'],
        num_rows: 8202
    })
})


In [12]:
df = ds["train"].to_pandas()
df.head()

,id,text,language,source_dataset,is_scam,head1_social_engineering,head2_scam_intent,split_group,metadata
0,real_051067,"Oh so you were late?? I went aye also, dun rmb...",English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
1,real_074028,"?? Gd night, sleep well!",English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
2,real_074336,It wo n't be long until you receive your Mand ...,English,Smishing_Dataset,1,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Courier Scam,real_smishing_dataset,"{'has_url': True, 'has_phone': False, 'has_otp..."
3,real_033886,Which is now right.,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
4,real_001118,I'm not driving... Raining! Then i'll get caug...,English,UCI_SMS_Spam,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_uci_sms_spam,"{'has_url': False, 'has_phone': False, 'has_ot..."


In [13]:
df["source_dataset"].value_counts()

source_dataset
Smishing_Dataset                      55475
UCI_SMS_Spam                           4175
Synthetic_Tier_C                       3720
Kaggle_Hindi_Merged                    3652
Indian_Telecom_SMS                     1627
ysangam/Indian_Cyber_Scam_Hinglish      597
Benchmark_Ground_Truth                   40
Name: count, dtype: int64

In [14]:
df.isnull().sum()

id                          0
text                        0
language                    0
source_dataset              0
is_scam                     0
head1_social_engineering    0
head2_scam_intent           0
split_group                 0
metadata                    0
dtype: int64

In [15]:
df.duplicated(subset="text").sum()

np.int64(0)

In [16]:
train = ds["train"].to_pandas()
val   = ds["validation"].to_pandas()
test  = ds["test"].to_pandas()

# keep real messages only (synthetic ids start with "synth_")
train_real = train[train["id"].str.startswith("real_")]
val_real   = val[val["id"].str.startswith("real_")]
test_real  = test[test["id"].str.startswith("real_")]

print(len(train_real), len(val_real), len(test_real))
print(train_real["is_scam"].value_counts())

65566 8192 8202
is_scam
0    41505
1    24061
Name: count, dtype: int64


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),   # single words and word pairs
    min_df=2,             # ignore words that appear only once
    sublinear_tf=True,
)

X_train = tfidf.fit_transform(train_real["text"])   # learn from train only
X_val   = tfidf.transform(val_real["text"])
X_test  = tfidf.transform(test_real["text"])

y_train = train_real["is_scam"]
y_val   = val_real["is_scam"]
y_test  = test_real["is_scam"]

print(X_train.shape)

(65566, 130145)


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

print(classification_report(y_val, model.predict(X_val)))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98      5186
           1       0.95      0.97      0.96      3006

    accuracy                           0.97      8192
   macro avg       0.97      0.97      0.97      8192
weighted avg       0.97      0.97      0.97      8192



In [19]:
val2 = val.assign(pred=model.predict(X_val))
print(val2.groupby("source_dataset").apply(lambda g: (g["pred"] == g["is_scam"]).mean()))
print(val2["source_dataset"].value_counts())

source_dataset
Benchmark_Ground_Truth                0.000000
Indian_Telecom_SMS                    0.973545
Kaggle_Hindi_Merged                   0.982418
Smishing_Dataset                      0.967937
UCI_SMS_Spam                          0.978599
ysangam/Indian_Cyber_Scam_Hinglish    1.000000
dtype: float64
source_dataset
Smishing_Dataset                      6955
UCI_SMS_Spam                           514
Kaggle_Hindi_Merged                    455
Indian_Telecom_SMS                     189
ysangam/Indian_Cyber_Scam_Hinglish      76
Benchmark_Ground_Truth                   3
Name: count, dtype: int64


C:\Users\adity\AppData\Local\Temp\ipykernel_29816\3119648808.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(val2.groupby("source_dataset").apply(lambda g: (g["pred"] == g["is_scam"]).mean()))


In [20]:
val2[val2["source_dataset"] == "Benchmark_Ground_Truth"][["text", "is_scam", "pred"]]

,text,is_scam,pred
2448,"Study in UK, USA, Canada, Australia, Malaysia,...",0,1
3436,NEW ARRIVAL - JUNE 23RD\n\nDresses @ 300; Kond...,0,1
8186,बैंक अलर्ट Yesterday 11:30 PM\nऋण विभाग से सूच...,1,0


In [21]:
overlap = val_real["text"].str.lower().str.strip().isin(train_real["text"].str.lower().str.strip())
print(overlap.mean())

0.0


In [22]:
errs = val2[(val2["pred"] != val2["is_scam"]) & (val2["source_dataset"] == "Indian_Telecom_SMS")]
errs[["text", "is_scam", "pred"]].head(15)

,text,is_scam,pred
895,Ã°ÂÂ Â°Ã¯Â¸ÂNifty 50 rose 173 points this we...,0,1
1613,""" Hello everyone, I am Arvind Singh the assist...",0,1
1879,Guys please join handover call,0,1
4229,Join from,0,1
6257,Hello! We love that you found what you were lo...,1,0


In [23]:
errs_all = val2[val2["pred"] != val2["is_scam"]]
errs_all[["source_dataset", "text", "is_scam", "pred"]].sample(30, random_state=1)

,source_dataset,text,is_scam,pred
2380,Smishing_Dataset,Wow! Airtel has given you 5mins FREE Airtel to...,0,1
8191,Kaggle_Hindi_Merged,औद्योगीकृत - 2904973 मीटर की दूरी पर. आपका प्र...,1,0
7226,Smishing_Dataset,WCF invites U 2 b our special guest @ 2day's H...,1,0
5203,Smishing_Dataset,Part 1 Course at Gyan manch & Shankari Bhawan ...,1,0
3187,Smishing_Dataset,"Iiii dunno. User your best judgement, we've be...",0,1
7043,Smishing_Dataset,I now can sleep through the night with no pain...,1,0
1967,Smishing_Dataset,You can just call when you get to your house :),0,1
7315,Smishing_Dataset,negro died went heaven angelwho r u negro imp...,1,0
5832,Smishing_Dataset,CALL ME URGENT ............... BIG MOVEMENT IN...,1,0
160,Smishing_Dataset,"_/L_ ( )Safety _Â¤_ ( )Blessings _*_ ("" )Pe...",0,1


In [24]:
pd.crosstab(train_real["head2_scam_intent"], train_real["is_scam"])

is_scam,0,1
head2_scam_intent,,
Bank Account Freeze,0,107
Courier Scam,0,764
Electricity Disconnection,0,124
Fake KYC,0,1026
General Spam / Telemarketing,0,19439
Legitimate / Benign,41505,0
Loan Scam,0,368
Lottery / Prize,0,640
OTP / Credential Theft,0,79


In [25]:
def make_label(d):
    def f(s):
        if s == "Legitimate / Benign":
            return "normal"
        if s == "General Spam / Telemarketing":
            return "spam"
        return "fraud"
    return d["head2_scam_intent"].map(f)

y_train3 = make_label(train_real)
y_val3   = make_label(val_real)

model3 = LogisticRegression(max_iter=1000, class_weight="balanced")
model3.fit(X_train, y_train3)
print(classification_report(y_val3, model3.predict(X_val)))

              precision    recall  f1-score   support

       fraud       0.96      0.93      0.94       566
      normal       0.98      0.97      0.98      5186
        spam       0.93      0.96      0.95      2440

    accuracy                           0.96      8192
   macro avg       0.96      0.95      0.96      8192
weighted avg       0.96      0.96      0.96      8192



In [26]:
from sklearn.metrics import confusion_matrix
labels = ["fraud", "normal", "spam"]
print(confusion_matrix(y_val3, model3.predict(X_val), labels=labels))

[[ 524   10   32]
 [   9 5029  148]
 [  11   82 2347]]


In [27]:
v = val_real.assign(pred=model3.predict(X_val), true=y_val3)
pd.set_option("display.max_colwidth", 200)
v[(v["true"] == "fraud") & (v["pred"] != "fraud")][["text", "pred"]].head(20)

,text,pred
4800,easy low cost tempo truck hiring and packers service by trukkycom call us for assistance at or download our mobile app bitlytrukkyandroid,spam
4948,central bank of india offers festival bonanza zero processing charges and reduced interest rate on retail loans valid upto dec tnc apply,spam
5037,Your package be be processe in Aramex warehouse Please note that confirmation of the shipping fee 3.30USD,spam
5038,RMO unifi TV WATCH WIN fab prize every week Do n't miss out this opportunity For more information i.unifi.my/tv,normal
5060,Dear Customer as per your request your Mobile number be Changed fror .. to XXXXXXXXXXXX kindly seed your Aadhaar no CBol,spam
5062,Goodmorning i ve be try ur line network be bad Am EMMANUEL OBINNA Ur EX KOPA MATE OSUN STATE my inlaw secure me a job at Shell spdc internal recruitment be go on now call me if u will apply,normal
5091,"Dear customer congratulation you have product order from CLUBFACTORY.COM PVT.LTD TV SHOW out in CLUBFACTORY.COM PVT.LTD COMPANY your name to open lucky draw MAHINDRA XUV500 CAR cost RS 16,96,000.0...",spam
5153,hi customer here is the exclusive offer only for you buy tees only link httpysmrtlnkinnosotpk offer valid for hours,spam
5163,The last thing NSW need be big immigration and rise power price One Nation will cut immigration number and electricity bill Learn More bit.ly/nswauth,spam
5230,"This be an example of how hacker can steal your credential include OTP.A charge of 1,200 be make to your GRAB PAY account If unauthorised visit grab-pay auth.com now",spam


In [28]:
tests = [
    "Dear customer, your SBI account will be blocked today. Update KYC now: bit.ly/3xYz",
    "Rs 500 debited from A/c XX1234 on 28-09-26. UPI Ref 123456. Not you? Call 1800-XXX",
    "123456 is your OTP for login. Do not share it with anyone.",
    "Congratulations! You won Rs 25 lakh in KBC lottery. Send your details to claim.",
    "Hey, are we still meeting for lunch tomorrow?",
]
print(list(zip(tests, model3.predict(tfidf.transform(tests)))))

[('Dear customer, your SBI account will be blocked today. Update KYC now: bit.ly/3xYz', 'fraud'), ('Rs 500 debited from A/c XX1234 on 28-09-26. UPI Ref 123456. Not you? Call 1800-XXX', 'spam'), ('123456 is your OTP for login. Do not share it with anyone.', 'fraud'), ('Congratulations! You won Rs 25 lakh in KBC lottery. Send your details to claim.', 'fraud'), ('Hey, are we still meeting for lunch tomorrow?', 'normal')]


In [29]:
import re
strip = lambda s: re.sub(r"\b(otp|upi)\b", " ", s, flags=re.I)

X_train2 = tfidf.fit_transform(train_real["text"].map(strip))
X_val2   = tfidf.transform(val_real["text"].map(strip))

model_b = LogisticRegression(max_iter=1000, class_weight="balanced")
model_b.fit(X_train2, y_train3)
print(classification_report(y_val3, model_b.predict(X_val2)))
print(model_b.predict(tfidf.transform([strip(t) for t in tests])))

              precision    recall  f1-score   support

       fraud       0.96      0.93      0.94       566
      normal       0.98      0.97      0.98      5186
        spam       0.93      0.96      0.94      2440

    accuracy                           0.96      8192
   macro avg       0.96      0.95      0.95      8192
weighted avg       0.96      0.96      0.96      8192

['fraud' 'spam' 'fraud' 'fraud' 'normal']


In [30]:
real_tests = [
    ("Rs 1,250.00 debited from A/c XX7305 on 27-09-26. Avl Bal Rs 8,420. Not you? Call 1800-000-000", "normal"),
    ("482913 is your OTP for login to the app. Do not share it with anyone.", "normal"),
    ("Your order has been shipped and will arrive by Tuesday.", "normal"),
    ("Flat 40% off on all shoes this weekend! Visit our store or shop online.", "spam"),
    ("Your KYC is expired. Update now to avoid account block: bit.ly/xyz123", "fraud"),
]

texts    = [t for t, _ in real_tests]
expected = [e for _, e in real_tests]

X = tfidf.transform([strip(t) for t in texts])
pred  = model_b.predict(X)
probs = model_b.predict_proba(X)

res = pd.DataFrame({
    "text": [t[:60] for t in texts],
    "expected": expected,
    "predicted": pred,
    "confidence": probs.max(axis=1).round(2),
})
res["correct"] = res["expected"] == res["predicted"]

pd.set_option("display.max_colwidth", 70)
print(res)
print("\nAccuracy:", res["correct"].mean().round(2))
print(res.groupby("expected")["correct"].mean().round(2))

                                                           text expected  \
0  Rs 1,250.00 debited from A/c XX7305 on 27-09-26. Avl Bal Rs    normal   
1  482913 is your OTP for login to the app. Do not share it wit   normal   
2       Your order has been shipped and will arrive by Tuesday.   normal   
3  Flat 40% off on all shoes this weekend! Visit our store or s     spam   
4  Your KYC is expired. Update now to avoid account block: bit.    fraud   

  predicted  confidence  correct  
0      spam        0.77    False  
1     fraud        0.88    False  
2      spam        0.51    False  
3      spam        0.98     True  
4     fraud        0.96     True  

Accuracy: 0.4
expected
fraud     1.0
normal    0.0
spam      1.0
Name: correct, dtype: float64


In [31]:
real_tests = [
    
 

    # ---- Promotions from real companies (spam) ----
    ("Flat 40% off on all footwear this weekend at Metro Shoes! Visit your nearest store. T&C apply.", "spam"),
    ("Big Billion Days starts tomorrow! Get up to 80% off on electronics. Shop now on the app.", "spam"),
    ("Get 5GB extra data on your next Airtel recharge of Rs 399. Recharge now on the Airtel Thanks app.", "spam"),
    ("Dominos: Buy 1 get 1 free on medium pizzas today only. Order on the app or call 1800-000-000.", "spam"),
    ("Diwali Sale at Reliance Digital: exchange your old TV and save up to Rs 10,000. Offer valid till 12-Oct-26.", "spam"),
    ("Renew your health policy with Star Health and get 10% off. Call 1800-000-000 for a free quote.", "spam"),

    # ---- Scam messages (fraud) ----
    ("Dear customer, your SBI account will be blocked today. Update your KYC now: bit.ly/3xYz", "fraud"),
    ("Congratulations! You have won Rs 25,00,000 in the KBC lottery. Send your name and bank details to claim your prize.", "fraud"),
    ("Your electricity will be disconnected tonight at 9:30 PM due to unpaid bill. Call 9876512345 immediately.", "fraud"),
    ("Your parcel is held at customs. Pay Rs 49 delivery fee at http://india-post-track.example to release it.", "fraud"),
    ("Part-time job: earn Rs 5,000 per day working from home. Join our Telegram group now: t.me/earnfast_jobs", "fraud"),
    ("Your PAN card is suspended. Update details within 24 hours at http://pan-verify-update.example or your account will be frozen.", "fraud"),
    ("Dear user, your HDFC credit card reward points worth Rs 9,800 will expire today. Redeem now: http://hdfc-rewards-claim.example", "fraud"),
    ("Traffic e-challan of Rs 2,000 issued for vehicle MH12AB1234. Pay within 24 hours to avoid court action: http://echallan-pay.example", "fraud"),
    ("Instant loan approved up to Rs 5,00,000 without documents! Pay processing fee Rs 999 to receive money in 10 minutes. Call 9876500000.", "fraud"),
    ("Your Amazon account has been locked due to suspicious activity. Verify your identity here: http://amazon-secure-login.example", "fraud"),
]
texts    = [t for t, _ in real_tests]
expected = [e for _, e in real_tests]

X = tfidf.transform([strip(t) for t in texts])
pred  = model_b.predict(X)
probs = model_b.predict_proba(X)

res = pd.DataFrame({
    "text": [t[:60] for t in texts],
    "expected": expected,
    "predicted": pred,
    "confidence": probs.max(axis=1).round(2),
})
res["correct"] = res["expected"] == res["predicted"]

pd.set_option("display.max_colwidth", 70)
print(res)
print("\nAccuracy:", res["correct"].mean().round(2))
print(res.groupby("expected")["correct"].mean().round(2))


                                                            text expected  \
0   Flat 40% off on all footwear this weekend at Metro Shoes! Vi     spam   
1   Big Billion Days starts tomorrow! Get up to 80% off on elect     spam   
2   Get 5GB extra data on your next Airtel recharge of Rs 399. R     spam   
3   Dominos: Buy 1 get 1 free on medium pizzas today only. Order     spam   
4   Diwali Sale at Reliance Digital: exchange your old TV and sa     spam   
5   Renew your health policy with Star Health and get 10% off. C     spam   
6   Dear customer, your SBI account will be blocked today. Updat    fraud   
7   Congratulations! You have won Rs 25,00,000 in the KBC lotter    fraud   
8   Your electricity will be disconnected tonight at 9:30 PM due    fraud   
9   Your parcel is held at customs. Pay Rs 49 delivery fee at ht    fraud   
10  Part-time job: earn Rs 5,000 per day working from home. Join    fraud   
11  Your PAN card is suspended. Update details within 24 hours a    fraud   

In [32]:
import re, pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit

def clean(s):
    s = str(s)
    s = re.sub(r"password\s*:\s*\S+", " passwordtoken ", s, flags=re.I)
    s = re.sub(r"https?://\S+|www\.\S+|\b(?:bit\.ly|t\.me|wa\.me)/\S+", " urltoken ", s, flags=re.I)
    s = re.sub(r"[xX*]{2,}", " maskedtoken ", s)
    s = re.sub(r"\d+", " numtoken ", s)
    return re.sub(r"\s+", " ", s).strip().lower()

mine = pd.read_csv("valid_messages.csv", encoding="utf-8")
mine = mine.rename(columns={"Message Content": "text"})
n0 = len(mine)
mine = mine.drop_duplicates("text")
n1 = len(mine)
mine = mine[~mine["text"].str.contains("wa.me", case=False)]   # the job-scam-looking message
mine = mine[~mine["text"].str.contains("👇")]                  # WhatsApp-style fragments
mine = mine[mine["text"].str.len() >= 40]                      # very short fragments
mine["label"] = "transactional"
mine["group"] = mine["text"].map(clean).str[:30]               # same template = same group
print(n0, "rows ->", n1, "unique ->", len(mine), "kept")

65 rows -> 49 unique -> 43 kept


In [33]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
tr, te = next(gss.split(mine, groups=mine["group"]))
my_train, my_test = mine.iloc[tr], mine.iloc[te]
print(len(my_train), "train |", len(my_test), "held-out")

30 train | 13 held-out


In [34]:
import pandas as pd, numpy as np
import datasets as hf

ds = hf.load_dataset("sidzzz07/scamshield-dataset")
train = ds["train"].to_pandas()
val   = ds["validation"].to_pandas()

# real messages only (same filter as before)
train_real = train[train["id"].str.startswith("real_")]
val_real   = val[val["id"].str.startswith("real_")]

# three classes: normal / spam / fraud
def to3(s):
    if s == "Legitimate / Benign":
        return "normal"
    if s == "General Spam / Telemarketing":
        return "spam"
    return "fraud"

y_train3 = train_real["head2_scam_intent"].map(to3)
y_val3   = val_real["head2_scam_intent"].map(to3)

print(len(train_real), len(val_real))
print(y_train3.value_counts())
print("real_tests defined:", "real_tests" in globals())

65566 8192
head2_scam_intent
normal    41505
spam      19439
fraud      4622
Name: count, dtype: int64
real_tests defined: True


In [36]:
for n in ["train_real", "y_train3", "my_train", "my_test", "clean", "real_tests", "tf4", "m4"]:
    print(n, n in globals())

train_real True
y_train3 True
my_train True
my_test True
clean True
real_tests True
tf4 False
m4 False


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ---- train ----
train_text = pd.concat([train_real["text"], my_train["text"]])
train_y    = pd.concat([y_train3, my_train["label"]])

tf4 = TfidfVectorizer(preprocessor=clean, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X4 = tf4.fit_transform(train_text)

m4 = LogisticRegression(max_iter=1000, class_weight="balanced")
m4.fit(X4, train_y)
print("classes:", list(m4.classes_))

# ---- test 1: held-out messages from your file ----
p = m4.predict(tf4.transform(my_test["text"]))
print("Held-out from your file (should be transactional):", (p == "transactional").mean().round(2))

# ---- test 2: the 42 messages ----
texts = [t for t, _ in real_tests]
exp   = ["transactional" if e == "normal" else e for _, e in real_tests]
pred  = m4.predict(tf4.transform(texts))
res = pd.DataFrame({"text": [t[:55] for t in texts], "expected": exp, "predicted": pred})
res["correct"] = res["expected"] == res["predicted"]
print(res.groupby("expected")["correct"].mean().round(2))
print("overall:", res["correct"].mean().round(2))
print(res[~res["correct"]])

# ---- test 3: dataset validation ----
print(classification_report(y_val3, m4.predict(tf4.transform(val_real["text"]))))

classes: ['fraud', 'normal', 'spam', 'transactional']
Held-out from your file (should be transactional): 0.38
expected
fraud    0.8
spam     1.0
Name: correct, dtype: float64
overall: 0.88
                                                       text expected  \
12  Dear user, your HDFC credit card reward points worth Rs    fraud   
15  Your Amazon account has been locked due to suspicious a    fraud   

   predicted  correct  
12      spam    False  
15      spam    False  
               precision    recall  f1-score   support

        fraud       0.96      0.92      0.94       566
       normal       0.98      0.97      0.98      5186
         spam       0.93      0.96      0.94      2440
transactional       0.00      0.00      0.00         0

     accuracy                           0.96      8192
    macro avg       0.72      0.71      0.71      8192
 weighted avg       0.96      0.96      0.96      8192



C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [43]:
import re, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import datasets as hf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

# ---------- 1. text cleaner ----------
def clean(s):
    s = str(s)
    s = re.sub(r"password\s*:\s*\S+", " passwordtoken ", s, flags=re.I)
    s = re.sub(r"https?://\S+|www\.\S+|\b(?:bit\.ly|t\.me|wa\.me)/\S+", " urltoken ", s, flags=re.I)
    s = re.sub(r"[xX*]{2,}", " maskedtoken ", s)
    s = re.sub(r"\d+", " numtoken ", s)
    return re.sub(r"\s+", " ", s).strip().lower()

# ---------- 2. ScamShield (real rows, 3 classes) ----------
ds = hf.load_dataset("sidzzz07/scamshield-dataset")
train = ds["train"].to_pandas()
train_real = train[train["id"].str.startswith("real_")]

def to3(s):
    if s == "Legitimate / Benign":
        return "normal"
    if s == "General Spam / Telemarketing":
        return "spam"
    return "fraud"

y_train3 = train_real["head2_scam_intent"].map(to3)
print("ScamShield real rows:", len(train_real))

# ---------- 3. your valid_messages.csv (label = transactional) ----------
mine = pd.read_csv("valid_messages.csv", encoding="utf-8").rename(columns={"Message Content": "text"})
mine = mine.drop_duplicates("text")
mine = mine[~mine["text"].str.contains("wa.me", case=False)]
mine = mine[~mine["text"].str.contains("👇")]
mine = mine[mine["text"].str.len() >= 40]
mine["label"] = "transactional"
mine["group"] = mine["text"].map(clean).str[:30]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
tr, te = next(gss.split(mine, groups=mine["group"]))
my_train, my_test = mine.iloc[tr], mine.iloc[te]
print("Your messages: train", len(my_train), "| held-out", len(my_test))

# ---------- 4. train ----------
train_text = pd.concat([train_real["text"], my_train["text"]])
train_y    = pd.concat([y_train3, my_train["label"]])

tf4 = TfidfVectorizer(preprocessor=clean, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X4  = tf4.fit_transform(train_text)
m4  = LogisticRegression(max_iter=1000, class_weight="balanced")
m4.fit(X4, train_y)
print("Trained. Classes:", list(m4.classes_))

# ---------- 5. real-message check ----------
my_real = [
    ("HDFC Bank: Rs 2,340.00 debited from a/c **7305 on 28-09-26 at BIGBASKET. Avl bal Rs 15,220.41. Not you? Call 1800-000-000", "transactional"),
    ("A/c 2096 credited Rs.32,000.00 on 26-09-26 by NEFT-ABC TECH SERVICES. Avl Bal Rs.58,114.20 -Axis Bank", "transactional"),
    ("Rs 799.00 paid to NETFLIX via UPI on 23-09-26. UPI Ref 528193746502. Bal Rs 3,041.90 -Canara Bank", "transactional"),
    ("Dear Customer, your loan EMI of Rs 8,450 for a/c XX9901 is due on 05-10-26. Keep sufficient balance. -Bank of Baroda", "transactional"),
    ("592047 is your OTP to log in to Flipkart. It is valid for 10 minutes. Do not share it with anyone.", "transactional"),
    ("OTP for your UPI transaction of Rs 3,200 is 480915. Do not share this with anyone. -Axis Bank", "transactional"),
    ("Your Blinkit order #7391842 is on the way and will arrive in 9 minutes.", "transactional"),
    ("Recharge of Rs 239 done for 9876512345. Validity 24 days, 1.5 GB/day. Thank you for choosing Vi.", "transactional"),
    ("Beware of fraud calls asking for your card details, PIN or OTP. The bank will never ask for these. -Union Bank of India", "transactional"),
    ("Bhai kal ka match dekha? Kohli ne kya khela yaar", "normal"),
    ("Mom, I will reach home by 8. Don't wait for dinner.", "normal"),
    ("Get flat Rs 500 off on your first order above Rs 1,499 at Ajio. Use code NEW500. Shop now on the app.", "spam"),
    ("Buy 2 get 1 free on all Lakme products this week only at Nykaa. Offer ends Sunday!", "spam"),
    ("Mega Sale at Croma! Up to 60% off on ACs and refrigerators. Visit your nearest store today.", "spam"),
    ("Planning a home loan? Get rates starting at 8.5%. Call 9876500000 for a free consultation.", "spam"),
    ("Dear customer your HDFC net banking will be deactivated today. Click http://hdfc-netbank-update.example to verify PAN now.", "fraud"),
    ("Your SBI YONO account has been suspended. To reactivate, update KYC here: http://sbi-yono-kyc.example", "fraud"),
    ("Congratulations! You are selected for Rs 10,000 cashback from Paytm. Claim within 1 hour: http://paytm-cashback-claim.example", "fraud"),
    ("Work from home: like YouTube videos and earn Rs 3,000 daily. Message us on WhatsApp 9876500000 to start. Limited seats.", "fraud"),
    ("Your Jio SIM will be blocked in 2 hours because your KYC is incomplete. Call 9876500000 now to avoid disconnection.", "fraud"),
    ("Dear customer, your parcel could not be delivered due to incorrect address. Update within 12 hours: http://india-post-redeliver.example", "fraud"),
    ("Income tax refund of Rs 24,850 is pending. Submit your bank details to receive it: http://incometax-refund-claim.example", "fraud"),
    ("Papa accident ho gaya hai, hospital mein hun. Abhi Rs 20,000 is UPI par bhejo 9876500000@ybl, baad mein batata hun.", "fraud"),
    ("You have won a brand new iPhone 16 in the Flipkart lucky draw. Pay Rs 99 delivery charge to claim: http://flipkart-lucky-win.example", "fraud"),
]

texts    = [t for t, _ in my_real]
expected = [e for _, e in my_real]
probs    = m4.predict_proba(tf4.transform(texts))
classes  = list(m4.classes_)
order    = np.argsort(-probs, axis=1)
pred     = [classes[i[0]] for i in order]
conf     = probs[np.arange(len(probs)), order[:, 0]]

res = pd.DataFrame({"text": [t[:60] for t in texts], "expected": expected,
                    "predicted": pred, "confidence": conf.round(2)})
res["correct"] = res["expected"] == res["predicted"]

pd.set_option("display.max_colwidth", 65, "display.width", 200, "display.max_rows", 200)
print("\n", res.to_string())
print("\nOverall accuracy:", res["correct"].mean().round(2))
print(res.groupby("expected")["correct"].mean().round(2))
print("\nWrong ones:")
print(res[~res["correct"]].to_string())
danger = res[(res["expected"] == "fraud") & (res["predicted"].isin(["transactional", "normal"]))]
print("\nDANGEROUS (scam called safe):", len(danger))

# your held-out messages from valid_messages.csv (should all be transactional)
ph = m4.predict(tf4.transform(my_test["text"]))
print("Held-out from your file predicted transactional:", (ph == "transactional").mean().round(2), f"({len(my_test)} msgs)")

ScamShield real rows: 65566
Your messages: train 30 | held-out 13
Trained. Classes: ['fraud', 'normal', 'spam', 'transactional']

                                                             text       expected      predicted  confidence  correct
0   HDFC Bank: Rs 2,340.00 debited from a/c **7305 on 28-09-26 a  transactional           spam        0.66    False
1   A/c 2096 credited Rs.32,000.00 on 26-09-26 by NEFT-ABC TECH   transactional           spam        0.66    False
2   Rs 799.00 paid to NETFLIX via UPI on 23-09-26. UPI Ref 52819  transactional          fraud        0.67    False
3   Dear Customer, your loan EMI of Rs 8,450 for a/c XX9901 is d  transactional          fraud        0.94    False
4   592047 is your OTP to log in to Flipkart. It is valid for 10  transactional  transactional        0.98     True
5   OTP for your UPI transaction of Rs 3,200 is 480915. Do not s  transactional  transactional        0.63     True
6   Your Blinkit order #7391842 is on the way and will ar

In [44]:
model.save("mal_email_detector.keras")

import joblib
joblib.dump(tfidf, "tfidf_vectorizer.joblib")

AttributeError: 'LogisticRegression' object has no attribute 'save'